# 🧠 MIND v3a — fine-tuning de MIND v2

On **repart de MIND v2 tel quel** (mêmes 69 M de paramètres, même architecture, 256 dimensions) et on continue son
entraînement avec deux améliorations, **sans rien changer d'autre** pour savoir exactement ce qu'elles apportent :

1. **Négatifs difficiles** : pour chaque phrase, on donne une phrase qui lui **ressemble mais ne veut pas dire la même chose**
   - XNLI : prémisse → implication ✅ / contradiction ❌ (« Le chat dort » vs « Le chat ne dort pas »)
   - PAWS-X : paraphrase ✅ / mêmes mots dans un autre ordre avec un autre sens ❌
2. **Gros lots** (512 au lieu de 64) grâce à `CachedMultipleNegativesRankingLoss` : plus de comparaisons par pas, même mémoire GPU

Uniquement les parties **train** des datasets (déjà vues par MIND v2), le **dev** STSb-fr sert à garder le meilleur point de contrôle,
le **test** n'est regardé qu'à la fin.

⚙️ Kaggle : **GPU T4**, **Internet ON**. Durée : ~30–60 min. Résultat : le dossier `mind-v3a/` (+ `mind-v3a.zip`) dans l'Output.
Ensuite : notebook 04 avec `NOM_MIND = "MIND v3a"` pour le comparer à v2 sur MTEB.

In [ ]:
!pip install -q -U "sentence-transformers[train]" datasets accelerate

In [ ]:
import os, glob, shutil, random, urllib.request
import numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset, DatasetDict
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
try:
    from sentence_transformers.sentence_transformer.training_args import BatchSamplers, MultiDatasetBatchSamplers
except ImportError:
    from sentence_transformers.training_args import BatchSamplers, MultiDatasetBatchSamplers

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SORTIE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
print("device :", DEVICE)

# ---- réglages ----
NOM_SORTIE = "mind-v3a"
N_TRIPLETS_XNLI = 150_000   # nombre max de triplets XNLI
TAILLE_LOT = 512            # lot « virtuel » (CachedMNRL le découpe en mini-lots)
MINI_LOT = 64               # ce qui passe réellement dans le GPU d'un coup
EPOQUES = 1
LR = 1e-5                   # plus bas que pour v2 (2e-5) : on affine un modèle déjà entraîné
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## MIND v2 (point de départ)

In [ ]:
FICHIERS_MIND = ["README.md", "config.json", "config_sentence_transformers.json", "modules.json",
                 "sentence_bert_config.json", "tokenizer.json", "tokenizer_config.json", "model.safetensors",
                 "1_Pooling/config.json", "2_Dense/config.json", "2_Dense/model.safetensors", "3_Normalize/config.json"]
LFS = "https://media.githubusercontent.com/media/LiamLitle/ia-de-liam/main/MIND/mind-v2-final/"
trouve = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/mind-v2-final/modules.json", recursive=True)]
if trouve:
    MIND_V2 = trouve[0]
else:
    MIND_V2 = f"{SORTIE}/mind-v2-final"
    for f in FICHIERS_MIND:
        dest = os.path.join(MIND_V2, f)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if not os.path.exists(dest):
            urllib.request.urlretrieve(LFS + f, dest)
modele = SentenceTransformer(MIND_V2, device=DEVICE)
print(modele)

## Données d'entraînement (parties train uniquement)

In [ ]:
# XNLI fr : 0 = implication, 1 = neutre, 2 = contradiction
xnli = load_dataset("facebook/xnli", "fr", split="train").to_pandas()
ent = xnli[xnli.label == 0].drop_duplicates("premise")[["premise", "hypothesis"]].rename(columns={"hypothesis": "positive"})
con = xnli[xnli.label == 2].drop_duplicates("premise")[["premise", "hypothesis"]].rename(columns={"hypothesis": "negative"})
triplets_xnli = ent.merge(con, on="premise").rename(columns={"premise": "anchor"})
triplets_xnli = triplets_xnli[triplets_xnli.anchor.str.len() > 15]
triplets_xnli = triplets_xnli.sample(min(N_TRIPLETS_XNLI, len(triplets_xnli)), random_state=SEED)[["anchor", "positive", "negative"]]

# PAWS-X fr : 1 = paraphrase, 0 = mêmes mots mais autre sens (négatif très difficile)
paws = load_dataset("google-research-datasets/paws-x", "fr", split="train").to_pandas()
paws = paws[(paws.sentence1.str.len() > 15) & (paws.sentence2.str.len() > 15)]
pos = paws[paws.label == 1].drop_duplicates("sentence1").rename(columns={"sentence1": "anchor", "sentence2": "positive"})
neg = paws[paws.label == 0].drop_duplicates("sentence1").rename(columns={"sentence1": "anchor", "sentence2": "negative"})
triplets_paws = pos[["anchor", "positive"]].merge(neg[["anchor", "negative"]], on="anchor")
paires_paws = pos[~pos.anchor.isin(triplets_paws.anchor)][["anchor", "positive"]]

train = DatasetDict({
    "xnli_triplets": Dataset.from_pandas(triplets_xnli, preserve_index=False),
    "paws_triplets": Dataset.from_pandas(triplets_paws, preserve_index=False),
    "paws_paires": Dataset.from_pandas(paires_paws, preserve_index=False),
})
print(train)
print(triplets_xnli.iloc[0].to_dict())

In [ ]:
from huggingface_hub import hf_hub_download
def stsb(split):
    d = pd.read_parquet(hf_hub_download("PhilipMay/stsb_multi_mt", f"fr/{split}-00000-of-00001.parquet", repo_type="dataset"))
    return EmbeddingSimilarityEvaluator(d.sentence1.tolist(), d.sentence2.tolist(), (d.similarity_score / 5).tolist(),
                                        name=f"stsb-fr-{split}")
eval_dev, eval_test = stsb("dev"), stsb("test")
avant = {"dev": eval_dev(modele), "test": eval_test(modele)}
print("MIND v2 — STSb-fr dev :", round(100 * avant["dev"]["stsb-fr-dev_spearman_cosine"], 2),
      "| test :", round(100 * avant["test"]["stsb-fr-test_spearman_cosine"], 2))

## Entraînement

In [ ]:
perte = losses.CachedMultipleNegativesRankingLoss(modele, mini_batch_size=MINI_LOT)
args = SentenceTransformerTrainingArguments(
    output_dir=f"{SORTIE}/checkpoints",
    num_train_epochs=EPOQUES,
    per_device_train_batch_size=TAILLE_LOT,
    learning_rate=LR,
    warmup_ratio=0.1,
    fp16=DEVICE == "cuda",
    batch_sampler=BatchSamplers.NO_DUPLICATES,          # pas deux fois la même phrase dans un lot
    multi_dataset_batch_sampler=MultiDatasetBatchSamplers.PROPORTIONAL,
    eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=50, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_stsb-fr-dev_spearman_cosine",
    logging_steps=10, report_to="none", seed=SEED,
)
entraineur = SentenceTransformerTrainer(model=modele, args=args, train_dataset=train, loss=perte, evaluator=eval_dev)
entraineur.train()

## Résultat et sauvegarde

In [ ]:
apres = {"dev": eval_dev(modele), "test": eval_test(modele)}
tab = pd.DataFrame({
    "MIND v2": [100 * avant["dev"]["stsb-fr-dev_spearman_cosine"], 100 * avant["test"]["stsb-fr-test_spearman_cosine"]],
    NOM_SORTIE: [100 * apres["dev"]["stsb-fr-dev_spearman_cosine"], 100 * apres["test"]["stsb-fr-test_spearman_cosine"]],
}, index=["STSb-fr dev", "STSb-fr test"]).round(2)
tab["écart"] = (tab[NOM_SORTIE] - tab["MIND v2"]).round(2)
print(tab)

DOSSIER = f"{SORTIE}/{NOM_SORTIE}"
modele.save(DOSSIER)
shutil.make_archive(DOSSIER, "zip", DOSSIER)
shutil.rmtree(f"{SORTIE}/checkpoints", ignore_errors=True)   # les points de contrôle prennent beaucoup de place
print("modèle sauvegardé :", DOSSIER, "et", DOSSIER + ".zip")
e = modele.encode(["Le chat dort sur le canapé.", "Un chat fait la sieste sur le sofa.", "Le chat ne dort pas sur le canapé."])
print(np.round(e @ e.T, 3))

### Et après ?
- Le STSb-fr **test** ne doit être regardé qu'une fois : c'est le verdict. Un gain sur le **dev** seul ne suffit pas.
- Pour le vrai verdict (25 tâches) : dans le notebook 04, **Add Input → l'Output de ce notebook**, puis
  `NOM_MIND = "MIND v3a"` et `MIND_DOSSIER = "/kaggle/input/<nom>/mind-v3a"` (adapter le chemin), `LEGER = False`.
- Regarde surtout **PairClassification / STS / Reranking** : c'est là que les négatifs difficiles doivent aider.